## Data diambil dari beberapa sumber (`sumber_id` di CSV)
| `sumber_id` | Keterangan | Level |
|---|---|---|
| `PIHPS` | PIHPS Bank Indonesia — **seri target** | Kota Bandung |
| `bisnis_pihps_jabar` | Berita Bisnis.com mengutip PIHPS | Jawa Barat |
| `pihps_berita` | Berita yang mengutip PIHPS | Kota Bandung |
| `pasar_berita` | Berita harga di pasar tradisional | Kota Bandung |
| `disdagin_berita` | Rilis/berita Disdagin Kota Bandung | Kota Bandung |
| `sp2kp_databoks` | SP2KP Kemendag (via Databoks) | Jawa Barat |
| `tepas_jabar_databoks` | Tepas Jabar (via Databoks), komoditas ikan | Jawa Barat |


# Prediksi Harga Bahan Baku UMKM Kuliner Kota Bandung
Hackathon Logicodix 2026 (subtema Local MSME). Notebook ini adalah *narasi*; logika ada di
modul `src/harga/` (dipakai ulang oleh backend app & dites di `tests/`).

Alur: **EDA → Cleaning → Seri target → Model + walk-forward → Prediksi Desember → Pipeline siap
data harian → Output app → Keterbatasan & Green IT.**

In [ ]:
import sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "harga").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from harga import config as C
from harga.load import load_csv, deteksi_frekuensi, median_gap_hari
from harga.clean import bersihkan, seri_target
from harga.models import backtest_bulanan, pilih_model
from harga.features import ke_harian, buat_fitur, fitur_kalender
from harga.predict import run

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
CSV = ROOT / "dataset" / "dataset_gabungan_komoditas_2026.csv"
df = load_csv(CSV)
print("ROOT:", ROOT, "| data:", df.shape)
df.head()

## Tahap 1 — EDA: membuktikan karakteristik data
Klaim yang diuji: (a) PIHPS bulanan, (b) SP2KP 1–3 titik/bulan mulai akhir April,
(c) nama dobel di PIHPS identik, (d) agregat = rata-rata subjenis, (e) nilai beku,
(f) duplikat tanggal+komoditas+sumber, (g) beda satuan & definisi antarsumber.

In [ ]:
# (a)(b) frekuensi & jarak antartanggal per sumber
rows = []
for s, g in df.groupby("sumber_id"):
    d = pd.Series(sorted(g["tanggal"].unique()))
    gap = d.diff().dt.days.dropna()
    rows.append({"sumber_id": s, "level": C.LEVEL_SUMBER.get(s), "baris": len(g),
                 "tanggal_unik": len(d), "awal": d.min().date(), "akhir": d.max().date(),
                 "gap_min": gap.min() if len(gap) else None,
                 "gap_median": gap.median() if len(gap) else None,
                 "gap_max": gap.max() if len(gap) else None})
display(pd.DataFrame(rows).sort_values("baris", ascending=False))
print("Frekuensi PIHPS terdeteksi:", deteksi_frekuensi(df.loc[df.sumber_id == "PIHPS", "tanggal"]))
print("Tanggal PIHPS:", [t.strftime("%Y-%m-%d") for t in sorted(df.loc[df.sumber_id == "PIHPS", "tanggal"].unique())])

In [ ]:
# Tabel jumlah titik per (komoditas, sumber_id)
display(pd.crosstab(df["komoditas"], df["sumber_id"], margins=True))

In [ ]:
# (c) nama dobel identik & (d) agregat = rata-rata subjenis (PIHPS)
p = df[df.sumber_id == "PIHPS"].pivot_table(index="tanggal", columns="komoditas", values="harga")
for alias, kanon in C.ALIAS_KANONIK.items():
    x = p[[alias, kanon]].dropna()
    print(f"{alias:28s} == {kanon:16s}: identik di {(x[alias] == x[kanon]).sum()}/{len(x)} titik")
print()
agregat = {"beras": [c for c in p if c.startswith("beras_kualitas")],
           "cabai_merah": ["cabai_merah_besar", "cabai_merah_keriting"],
           "cabai_rawit": ["cabai_rawit_hijau", "cabai_rawit_merah"],
           "gula_pasir": ["gula_pasir_lokal", "gula_pasir_kualitas_premium"],
           "minyak_goreng": ["minyak_goreng_curah", "minyak_goreng_kemasan_bermerk_1", "minyak_goreng_kemasan_bermerk_2"],
           "daging_sapi": ["daging_sapi_kualitas_1", "daging_sapi_kualitas_2"]}
for a, subs in agregat.items():
    print(f"{a:14s} vs rata-rata {len(subs)} subjenis: selisih maks Rp{(p[a] - p[subs].mean(axis=1)).abs().max():.0f}")

In [ ]:
# (e) nilai beku: porsi nilai modus & run terpanjang per seri SP2KP
def run_terpanjang(h):
    best = cur = 1
    for a, b in zip(h[:-1], h[1:]):
        cur = cur + 1 if a == b else 1
        best = max(best, cur)
    return best

sp = df[df.sumber_id == "sp2kp_databoks"].sort_values("tanggal")
beku = sp.groupby("komoditas")["harga"].agg(
    n="size", nilai_unik="nunique",
    porsi_modus=lambda h: h.value_counts().iloc[0] / len(h),
    run_terpanjang=lambda h: run_terpanjang(h.to_numpy()),
    nilai_modus=lambda h: h.mode().iloc[0])
display(beku.sort_values("porsi_modus", ascending=False).round(2))

In [ ]:
# (f) duplikat (tanggal, komoditas, sumber) & (g) satuan / jenis_harga / definisi beras
dup = df[df.duplicated(["tanggal", "komoditas", "sumber_id"], keep=False)]
print("Baris duplikat per sumber:"); display(dup.groupby(["sumber_id", "tanggal"]).size().rename("baris"))
print("Satuan per sumber:"); display(pd.crosstab(df["sumber_id"], df["satuan"]))
print("Baris non-eceran:"); display(df[df.jenis_harga != "eceran"])
print("Beras: SP2KP beras_medium vs PIHPS beras (agregat):")
display(pd.DataFrame({
    "SP2KP beras_medium (Jabar)": df.query("sumber_id=='sp2kp_databoks' and komoditas=='beras_medium'")["harga"].describe(),
    "PIHPS beras (Bandung)": df.query("sumber_id=='PIHPS' and komoditas=='beras'")["harga"].describe()}).round(0))

### Kesimpulan EDA
- **PIHPS = data BULANAN**: 11 titik (awal bulan, 1 Jan–2 Nov 2026), jarak 28–33 hari.
- **SP2KP** (rata-rata Jawa Barat): 21 tanggal sejak 28 Apr, jarak 1–20 hari. **Nilai beku** nyata:
  minyakita 15.700 di 69% titik; run pendek juga di tepung terigu, kedelai, minyak curah.
- **Nama dobel identik 11/11 titik**: bawang_merah, bawang_putih (+`_ukuran_sedang`), daging_ayam(_ras),
  telur_ayam(_ras). `beras`, `cabai_merah`, `cabai_rawit`, `gula_pasir`, `minyak_goreng`, `daging_sapi`
  = rata-rata subjenisnya (selisih ≤ Rp50, pembulatan).
- **Duplikat** hanya di berita: `pasar_berita` 19 Mar (4 pasar × 6 komoditas) dan `pihps_berita` 23 Okt
  (cabai_merah, 3 baris) → kolom pasar tidak ada, jadi diagregasi median/min/max.
- **Satuan**: minyak di SP2KP dalam liter, PIHPS dalam kg. Ada 1 baris harga *distributor*.
- **Definisi beda**: beras SP2KP ~13.100–13.400 vs PIHPS ~15.150–15.600 → seri **tidak digabung**.
- Sumber tambahan level **Jawa Barat**: `bisnis_pihps_jabar` (19 Feb) & `tepas_jabar_databoks` (ikan) → validasi saja.

## Tahap 2 — Cleaning (tidak ada data yang dibuang diam-diam)
1. **Nama kanonik**: alias identik dilebur ke satu nama (`bawang_merah`, `bawang_putih`, `daging_ayam_ras`, `telur_ayam_ras`).
2. **Seri turunan** `beras_medium` (PIHPS) = rata-rata `beras_kualitas_medium_i` & `_ii` → kolom `turunan=True`.
3. **Satuan**: dipakai apa adanya dari sumber (tanpa konversi densitas). PIHPS menerbitkan minyak goreng per kg,
   SP2KP per liter; kolom `satuan` ikut ditulis di output.
4. **Seri dipisah per `sumber_id`** + kolom `level` (kota_bandung / jawa_barat).
5. **Agregasi** per (tanggal, komoditas, sumber, satuan, jenis_harga): median + min + max + `n_obs`.
6. **Flag** (tidak dibuang): `is_frozen` (≥3 titik berturut sama), `is_outlier` (IQR×3 pada % perubahan
   **dan** |perubahan| ≥ 10%), `is_eceran` (baris distributor dikecualikan dari target).

In [ ]:
clean = bersihkan(df)
print("baris mentah:", len(df), "→ baris bersih:", len(clean))
display(clean.groupby(["sumber_id", "level"])[["is_frozen", "is_outlier", "turunan"]].sum())
print("Baris hasil agregasi multi-pasar:")
display(clean[clean.n_obs > 1][["tanggal", "komoditas", "sumber_id", "harga", "harga_min", "harga_max", "n_obs"]])
print("Contoh konversi satuan minyak (SP2KP):")
print("Outlier yang ditandai:")
display(clean[clean.is_outlier][["tanggal", "komoditas", "sumber_id", "harga", "pct_change"]].round({"pct_change": 3}))

Catatan: di PIHPS banyak `is_frozen` karena beras/gula/minyak memang jarang berubah antarbulan
(harga stabil/teregulasi) — flag ini informatif, bukan berarti salah. Di SP2KP, nilai beku lebih
mencurigakan (data mingguan yang tidak bergerak) sehingga SP2KP hanya dipakai sebagai pembanding.

## Tahap 3 — Komoditas target (`C.TARGET_KOMODITAS`) & seri target
Jumlah komoditas target = **13** (`len(C.TARGET_KOMODITAS)`): 8 utama (beras medium, gula pasir, minyak
goreng curah, telur ayam ras, daging ayam ras, cabai merah keriting, cabai rawit merah, bawang merah)
+ 5 tambahan dari PIHPS (bawang putih, cabai merah besar, cabai rawit hijau, daging sapi, gula pasir premium).
Target = **PIHPS Kota Bandung (bulanan)**. SP2KP (Jabar) dan berita (Bandung, titik lepas) hanya
**pendukung/validasi** — beda definisi & cakupan, jadi tidak digabung ke seri target.

In [ ]:
wide = seri_target(clean)
display(wide.astype(int))

In [ ]:
import math
import matplotlib.dates as mdates

koms = list(C.TARGET_KOMODITAS)
ncol = 3
nrow = math.ceil(len(koms) / ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(13, 3.2 * nrow), sharex=True)
axes = axes.ravel()

for ax, kom in zip(axes, koms):
    if kom in wide:
        ax.plot(wide.index, wide[kom], "o-", label="PIHPS Bandung (target)")
    s = clean[(clean.komoditas == kom) & (clean.sumber_id == "sp2kp_databoks")]
    if len(s): ax.plot(s.tanggal, s.harga, ".--", alpha=.7, label="SP2KP Jabar")
    b = clean[(clean.komoditas == kom) & clean.sumber_id.isin(["pasar_berita", "disdagin_berita", "pihps_berita", "bisnis_pihps_jabar"]) & clean.is_eceran]
    if len(b):
        ax.errorbar(b.tanggal, b.harga, yerr=[b.harga - b.harga_min, b.harga_max - b.harga],
                    fmt="s", ms=4, capsize=3, alpha=.8, label="Berita (median, min–max)")
    ax.set_title(kom, fontsize=10); ax.grid(alpha=.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1000:.0f}rb"))

# Sembunyikan panel kosong (sisa slot grid bila jumlah komoditas tidak habis dibagi ncol)
for ax in axes[len(koms):]:
    ax.set_visible(False)

# Format tanggal: label per bulan, singkat, tidak bertumpuk
for ax in axes:
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax.tick_params(axis="x", labelrotation=0, labelsize=8, labelbottom=True)

# Legend di atas gambar, tidak menutupi garis
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, 1.0))
fig.suptitle("Harga per kg — seri target vs pembanding (tidak digabung)", y=1.03)
fig.tight_layout()
plt.show()
print(f"{len(koms)} komoditas, grid {nrow}x{ncol}, {nrow*ncol-len(koms)} panel kosong disembunyikan")

In [ ]:
# Validasi: titik berita/Jabar vs PIHPS bulan yang sama (selisih %)
pend = clean[clean.komoditas.isin(C.TARGET_KOMODITAS) & (clean.sumber_id != "PIHPS") & clean.is_eceran].copy()
pihps_bln = wide.copy(); pihps_bln.index = pihps_bln.index.to_period("M")
pend["pihps_bulan_itu"] = [pihps_bln.at[t.to_period("M"), k] if t.to_period("M") in pihps_bln.index else np.nan
                           for t, k in zip(pend.tanggal, pend.komoditas)]
pend["selisih_pct"] = (pend.harga / pend.pihps_bulan_itu - 1) * 100
display(pend.groupby(["komoditas", "sumber_id", "level"])["selisih_pct"].agg(["count", "median", "min", "max"]).round(1))

## Tahap 4 — Model sesuai data yang ADA
Hanya **11 titik bulanan** per komoditas. Kandidat: **naive** (harga terakhir), **MA(3)**,
**drift** (tren garis lurus titik pertama–terakhir), **SES** (α dipilih dari grid hanya di data latih).

**Kenapa tidak LightGBM / SARIMA?** (aturan: < 30 titik → tidak dipakai)
- SARIMA musiman tahunan butuh ≥ 2 siklus (≥ 24 bulan) untuk mengestimasi komponen musim; dengan
  11 titik, parameter ARIMA tidak stabil dan pola Nataru/Lebaran belum pernah terlihat dua kali.
- LightGBM dengan lag 1–3 bulan hanya punya ±7 baris latih per komoditas → hafal noise (overfit),
  dan walk-forward hanya 7 titik uji sehingga keunggulannya tidak bisa dibuktikan.

**Validasi walk-forward**: latih s.d. bulan *t*, uji bulan *t+1*, mulai *t* = 4 → 7 titik uji per komoditas.
**Aturan pemilihan (margin 10%)**: model non-naive hanya dipilih bila MAE-nya **minimal
`C.MARGIN_MENANG_NAIVE` = 10% lebih kecil** dari naive. Dengan hanya 7 titik uji, selisih MAE 1–2%
tidak bisa dibedakan dari kebetulan — misalnya MA3 yang menang 2% atas naive untuk cabai bukan bukti
MA3 lebih baik. Kalau tidak menang dengan margin itu, dipakai naive (harga terakhir).

In [ ]:
print("Jumlah titik per komoditas:", wide.notna().sum().to_dict(), "| ambang ML:", C.MIN_TITIK_ML)
bt, residual = backtest_bulanan(wide)
print("MAPE walk-forward (%):"); display(bt.pivot(index="komoditas", columns="model", values="MAPE").round(2))
print("MAE walk-forward (Rp):"); display(bt.pivot(index="komoditas", columns="model", values="MAE").round(0))
terpilih = {k: pilih_model(bt[bt.komoditas == k]) for k in wide.columns}
display(pd.Series(terpilih, name="model_terpilih").to_frame())
mae = bt.pivot(index="komoditas", columns="model", values="MAE")
rasio = (mae.div(mae["naive"], axis=0)).round(3)
print(f"MAE relatif terhadap naive (pilih non-naive hanya bila <= {1 - C.MARGIN_MENANG_NAIVE:.2f}):")
display(rasio)
terpilih_lama = {k: pilih_model(bt[bt.komoditas == k], margin=0.0) for k in wide.columns}
display(pd.DataFrame({"aturan_lama (asal < naive)": terpilih_lama, "aturan_baru (margin 10%)": terpilih}))


Interpretasi: komoditas pokok (beras, gula, minyak) sangat stabil — naive/SES sudah ~0,1–2,6% MAPE;
telur & daging ayam ~4–5,5%. Cabai & bawang bergejolak (MAPE ~15–25%): tidak ada model sederhana yang andal. Dengan margin 10%,
hanya **cabai_rawit_merah (SES, MAE 0,70× naive)** dan **cabai_rawit_hijau (MA3, 0,82×)** yang benar-benar
mengalahkan naive; MA3/SES/drift yang dulu menang tipis (cabai keriting 0,98×, cabai merah besar 0,995×,
minyak 0,97×, telur 0,93×) kini kembali ke naive. Ini dikomunikasikan ke pedagang lewat rentang lebar,
bukan disembunyikan.

## Tahap 5 — Prediksi Desember 2026 + prediction interval
- Model terpilih dilatih ulang pada seluruh 11 titik, prediksi 1 bulan ke depan (awal Desember).
- **Interval** = prediksi + kuantil 10% & 90% residual walk-forward model itu (PI nominal 80%).
  Dengan hanya 7 residual, interval ini **kasar**; batas dipaksa mengapit prediksi.
- **Lebar minimum ±1%** (`C.LEBAR_MIN_INTERVAL`): untuk seri yang hampir tidak pernah berubah
  (beras, gula) residualnya banyak yang 0, sehingga kuantil memberi batas = prediksi. Itu bukan
  kepastian, hanya akibat sedikitnya titik — jadi batas dilebarkan minimal ±1% dari prediksi.
- **Nataru**: dataset dimulai Jan 2026, jadi belum ada Desember historis → efek musiman **tidak**
  diberi angka, hanya flag di kolom `catatan`.

In [ ]:
from harga.models import buat_model, interval_empiris
rows = []
for kom in wide.columns:
    y = wide[kom].dropna().to_numpy(); nama = terpilih[kom]
    pred = buat_model(nama).fit(y).predict(1)
    lo, hi = interval_empiris(pred, residual[(kom, nama)])
    rows.append({"komoditas": kom, "model": nama, "harga_nov": y[-1], "prediksi_des": pred,
                 "batas_bawah": lo, "batas_atas": hi, "perubahan_pct": (pred / y[-1] - 1) * 100,
                 "flag_nataru": True})
des = pd.DataFrame(rows); display(des.round(1))

fig, ax = plt.subplots(figsize=(10, 4))
rel = des.assign(lo=(des.batas_bawah / des.harga_nov - 1) * 100, hi=(des.batas_atas / des.harga_nov - 1) * 100)
ax.errorbar(rel.perubahan_pct, rel.komoditas, xerr=[rel.perubahan_pct - rel.lo, rel.hi - rel.perubahan_pct], fmt="o", capsize=4)
for x in (-5, 0, 5): ax.axvline(x, ls=":" if x else "-", c="grey", lw=1)
ax.set_xlabel("% perubahan Des vs Nov (titik = prediksi, garis = interval 80%)"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Tahap 6 — Pipeline SIAP DATA HARIAN
`run(csv)` mendeteksi frekuensi PIHPS secara otomatis:
- **bulanan** → baseline Tahap 4–5;
- **harian** → `ke_harian` (resample harian, ffill **maks 3 hari**, kolom `is_imputed`) →
  `buat_fitur` (lag 1/7/14, rolling 7/30, `day_of_week`, dummy Ramadan 18 Feb–19 Mar,
  Idulfitri 20–21 Mar, **Iduladha 27 Mei 2026** + cuti bersama 28 Mei — terverifikasi dari sidang isbat
  Kemenag/SKB 3 Menteri, Nataru 15 Des–5 Jan) → **LightGBM global** (semua komoditas, target = rasio
  perubahan) vs baseline naive & MA7, horizon **1 dan 7 hari**, walk-forward 4 fold × 14 hari.
  LightGBM **hanya dipakai bila MAE-nya mengalahkan baseline terbaik**.

Di bawah: fungsi harian dijalankan pada data ASLI (bulanan) untuk menunjukkan ffill tidak mengarang
data, lalu test otomatis (data harian sintetis *di dalam test saja*) membuktikan mode harian berjalan.

In [ ]:
target_long = clean[(clean.sumber_id == "PIHPS") & clean.komoditas.isin(C.TARGET_KOMODITAS)][["tanggal", "komoditas", "harga"]]
daily = ke_harian(target_long)
ringkas = daily.groupby("komoditas").agg(hari=("harga", "size"), terisi=("harga", "count"), imputed=("is_imputed", "sum"))
ringkas["kosong (celah > 3 hari, tidak dikarang)"] = ringkas.hari - ringkas.terisi
display(ringkas)
display(daily[daily.komoditas == "bawang_merah"].head(6))

In [ ]:
f = buat_fitur(daily, h=7)
print("Kolom fitur:", [c for c in f.columns if c.startswith(("lag", "r_", "roll", "is_", "day"))])
kal = fitur_kalender(pd.Series(pd.to_datetime(["2026-03-01", "2026-03-20", "2026-05-27", "2026-12-25", "2026-07-01"])))
kal.index = ["Ramadan", "Idulfitri", "Iduladha", "Natal", "biasa"]; display(kal)

In [ ]:
hasil_test = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", str(ROOT / "tests")],
                            capture_output=True, text=True, cwd=ROOT)
print(hasil_test.stdout[-1500:])
assert hasil_test.returncode == 0, hasil_test.stderr

## Tahap 7 — Output untuk aplikasi pedagang
Kolom: `komoditas | harga_terakhir | prediksi | batas_bawah | batas_atas | status | saran` + metadata
transparansi (`model`, `mape_validasi`, `n_titik`, `sumber`, `catatan`).
**Aturan status** (perubahan = prediksi / harga_terakhir − 1):
| Kondisi | Status |
|---|---|
| `mape_validasi` > 10% (`C.AMBANG_MAPE_BERGEJOLAK = 0.10`, dibandingkan dengan MAPE/100), **dicek paling dulu**, apa pun perubahannya | **BERGEJOLAK** |
| (selain di atas) \|perubahan\| < 5% | **STABIL** |
| \|perubahan\| ≥ 5% **dan** batas_bawah ≤ harga_terakhir ≤ batas_atas | **TIDAK PASTI** |
| perubahan ≥ +5% **dan** seluruh interval di atas harga terakhir | **NAIK** |
| perubahan ≤ −5% **dan** seluruh interval di bawah harga terakhir | **TURUN** |

Contoh: cabai merah keriting dulu "TURUN −22%, tunda pembelian", padahal intervalnya
(61.520–103.447) memuat harga terakhir 94.100 → model sendiri tidak yakin arahnya → **TIDAK PASTI** (dengan aturan terbaru: **BERGEJOLAK**, karena MAPE-nya 25%).

**Aturan saran**: BERGEJOLAK → selalu *"Harga bergejolak, beli sesuai kebutuhan"* tanpa arah
(tidak pernah "Beli seperti biasa"). Ambang 10% dipilih supaya bawang merah (MAPE 14,98%) ikut BERGEJOLAK.
TIDAK PASTI → *"Arah harga belum jelas, beli sesuai kebutuhan"*. Komoditas mudah rusak tidak disarankan ditimbun.

**Cache**: `run()` memakai ulang `output/prediksi.csv` hanya bila hash CSV **dan** hash kode `src/harga/*.py`
(termasuk `config.py`) sama dengan saat output dibuat. Mengubah konstanta/aturan otomatis memicu retrain.

**Isi `output/`**: satu `model_bulanan_<komoditas>.joblib` per komoditas di `C.TARGET_KOMODITAS`
(sekarang 13 = 8 utama + 5 tambahan, disengaja) + `prediksi.csv` + `cache_meta.json`. Model untuk
komoditas yang dikeluarkan dari target otomatis dihapus saat `run()`, jadi isi folder selalu sesuai target. Disimpan ke `output/prediksi.csv` dan `output/prediksi.json` (format untuk aplikasi web), model ke `output/*.joblib`.

In [ ]:
hasil = run(CSV, out_dir=ROOT / "output", paksa=True)
print("Mode:", hasil["mode"])
pred = hasil["prediksi"]
display(pred[["komoditas", "harga_terakhir", "prediksi", "batas_bawah", "batas_atas", "status", "saran"]])
display(pred[["komoditas", "model", "mape_validasi", "n_titik", "sumber", "catatan"]])
print(sorted(p.name for p in (ROOT / "output").iterdir()))
n_model = len(list((ROOT / "output").glob("model_bulanan_*.joblib")))
print(f"Model tersimpan: {n_model} | komoditas target: {len(C.TARGET_KOMODITAS)}")
assert n_model == len(C.TARGET_KOMODITAS)
display(pred["status"].value_counts().rename("jumlah"))
print("Run ulang tanpa perubahan CSV → dari cache:", run(CSV, out_dir=ROOT / "output")["dari_cache"])

## Tahap 8 — Keterbatasan, transparansi, Green IT

**Keterbatasan data**
- Data target **bulanan** (11 titik): prediksi saat ini = harga **awal bulan depan**, *bukan* besok/minggu
  depan. Tampilan "besok / 7 hari" di app baru valid setelah CSV diganti export PIHPS harian Kota Bandung —
  pipeline sudah otomatis beralih ke mode harian.
- Interval dari 7 residual → kasar; cabai & bawang MAPE ~15–23% → app menonjolkan rentang, bukan angka tunggal.
- Belum ada Desember historis → efek Nataru hanya flag, tidak diangkakan.
- **Prediksi Desember tidak memperhitungkan Nataru.** Model hanya melihat Jan–Nov, padahal menjelang
  Natal & Tahun Baru permintaan biasanya naik. Karena itu, arah **turun** untuk komoditas bergejolak
  (cabai, bawang; MAPE > 10% → status BERGEJOLAK) **tidak dijadikan saran** "tunda pembelian" — saran untuk komoditas itu
  selalu netral, dan komoditas lain berstatus TIDAK PASTI bila interval memuat harga terakhir.
- Densitas minyak 0,9 kg/L adalah asumsi (hanya memengaruhi seri pembanding SP2KP, bukan target).

**Transparansi sumber (AI yang bisa dijelaskan)**
- Setiap baris output membawa `sumber`, `model`, `mape_validasi`, `n_titik`, `catatan`.
- Sumber beda definisi/cakupan (PIHPS Bandung vs SP2KP Jabar vs berita) tidak dicampur; nilai beku & outlier
  diberi flag, tidak dihapus. Tidak ada data karangan: celah > 3 hari dibiarkan kosong.
- Privasi: data harga publik agregat, tanpa data pribadi pedagang.

**Green IT**
- Model ringan (naive/MA/SES/drift: milidetik di CPU; LightGBM kecil 200 pohon hanya di mode harian, dan
  hanya bila terbukti lebih baik dari baseline).
- **Cache**: `run()` menyimpan hash CSV + hash kode `src/harga/*.py` — bila data tidak berubah, hasil dibaca dari `output/prediksi.csv`
  tanpa retrain.
- **Retrain harian terjadwal** (mis. cron/Task Scheduler 1×/hari setelah PIHPS update), app hanya membaca
  CSV hasil → tidak ada inferensi per-request.